In [1]:
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import lime
import lime.lime_tabular

import matplotlib.pyplot as plt
import plotly.express as px

import huggingface_hub

import os
from dotenv import load_dotenv
import random
from tqdm import tqdm
import glob
from torch.utils.data import Dataset, DataLoader, random_split
from collections import OrderedDict
import soundfile as sf


from transformers import Wav2Vec2Model

from scipy.optimize import brentq
from scipy.interpolate import interp1d

from huggingface_hub import hf_hub_download

In [2]:
# load in huggingface authorization token
load_dotenv()
hf_auth=os.getenv("hf_auth")

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [45]:
class Wav2Vec2Deepfake(nn.Module):

    def __init__(self):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-base"
        )
        hidden_size = (
            self.wav2vec.config.hidden_size
        )
        for param in self.wav2vec.parameters():

            param.requires_grad = False
        for layer in self.wav2vec.encoder.layers[-2:]:

            for param in layer.parameters():

                param.requires_grad = True

        self.classifier = nn.Sequential(

            nn.Linear(hidden_size, 256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 2)
        )

    def forward(self, x):

        outputs = self.wav2vec(
            x
        )

        hidden_states = (
            outputs.last_hidden_state
        )

        pooled = hidden_states.mean(
            dim=1
        )

        logits = self.classifier(
            pooled
        )

        return logits

In [46]:
# import Wav2Vec Model from HuggingFace

w2v_model=Wav2Vec2Deepfake()
wav2vec_name='best_model.pth'
w2v_checkpoint=hf_hub_download(
    repo_id='rde6mn/no_aug_w2v_4s',
    filename=wav2vec_name
)
w2v_checkpoint=torch.load(w2v_checkpoint, map_location=device)
state_dict=w2v_checkpoint["model_state_dict"]

w2v_model.load_state_dict(state_dict)
w2v_model.eval()

# w2v parameters
SR = 16000
MAX_LEN = 4 * SR
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-5
NUM_WORKERS = 4

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
audiofile_dir=r"G:\My Drive\ASVSpoof_Data\unzipped2019\LA\LA\ASVspoof2019_LA_eval\flac"
audiofiles=glob.glob(os.path.join(audiofile_dir, "*.flac"))

def process_audio(audio_path):
    try:
        waveform, sr = sf.read(audio_path)
        if len(waveform.shape) > 1:
            waveform = waveform.mean(axis=1)

        if sr != SR:
            waveform = librosa.resample(
                waveform,
                orig_sr=sr,
                target_sr=SR
            )
    except Exception as e:
        try:
            waveform, sr = librosa.load(
                audio_path,
                sr=SR,
                mono=True
            )
        except Exception as e2:
            print(f"FAILED TO LOAD: {audio_path}")
            return None
    if len(waveform) > MAX_LEN:
        waveform = waveform[:MAX_LEN]
    else:
        waveform = np.pad(
            waveform,
            (0, MAX_LEN - len(waveform))
        )
    waveform = torch.tensor(
        waveform
    ).float()

    return waveform

In [48]:
waveforms=[]

# Preprocess the waveforms
for file in audiofiles:
    audio_path=file
    waveform=process_audio(audio_path)
    if waveform is not None:
        waveforms.append(waveform)
        if len(waveforms)==100:
                break
    else:
        print(f"FAILED TO LOAD: {audio_path}")
    


In [49]:
len(waveforms)

100

In [37]:
def generate_output(waveform):
    with torch.no_grad():

        waveform = waveform.unsqueeze(0)

        outputs = w2v_model(waveform)

        logits = outputs.logits

        probs = torch.softmax(logits, dim=1)[:, 1]

        preds = (probs >= 0.5).long()

    return probs, preds



In [38]:
wav0_output=generate_output(waveforms[0])
wav0_output

AttributeError: 'Wav2Vec2BaseModelOutput' object has no attribute 'logits'

In [ ]:
class Wav2Vec2ForLime(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.wav2vec = base_model
        self.classifier = nn.Linear(768, 2)  # 768 for base wav2vec2

    def forward(self, x):
        out = self.wav2vec(x).last_hidden_state  # [B, T, 768]

        pooled = out.mean(dim=1)  # [B, 768] (simple global average)

        logits = self.classifier(pooled)  # [B, 2]

        return torch.softmax(logits, dim=-1)
    
def predict_proba(waveforms):
    # waveforms: list/array of audio inputs OR batch tensor
    waveforms = torch.tensor(waveforms).float()

    if waveforms.dim() == 1:
        waveforms = waveforms.unsqueeze(0)

    with torch.no_grad():
        probs = w2v_model(waveforms)

    return probs.cpu().numpy()

In [50]:
w2v_model.eval()
predict_proba(waveforms[0])


C:\Users\eglha\AppData\Local\Temp\ipykernel_15780\3914727847.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveforms = torch.tensor(waveforms).float()


array([[0.52335566, 0.4766443 ]], dtype=float32)